In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
kagglehub.competition_download(
    'kmu-rec-sys-26-rating-prediction',
    output_dir = "dataset"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
!head dataset/train_small.csv

In [ ]:
!head dataset/test_small.csv

##데이터준비

In [ ]:
import csv
import torch
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

userIds, itemIds = {}, {}
users, items, ratings, dates = [], [], [], []

with open('dataset/train_small.csv',"r") as f:
  reader = csv.reader(f)
  next(reader)
  for userId, itemId, rating, date in reader:
    if userId not in userIds:
      userIds[userId] = len(userIds)
    if itemId not in itemIds:
      itemIds[itemId] = len(itemIds)

    users.append(userIds[userId])
    items.append(itemIds[itemId])
    ratings.append(float(rating))
    dates.append(datetime.strptime(date, "%Y-%m-%d"))

users = torch.tensor(users).to(device)
items = torch.tensor(items).to(device)
ratings = torch.tensor(ratings).to(device)

tmin = min(dates) # dates 날짜 중 가장 빠른 날짜
dates = [(t-tmin).days for t in dates]
dates = torch.tensor(dates, device=device)

In [ ]:
tmin

##데이터셋 분할

In [ ]:
n_train=int(len(ratings)*0.9)
indices=torch.randperm(len(ratings))
train_indices=indices[:n_train]
valid_indices=indices[n_train:]

users_valid=users[valid_indices]
items_valid=items[valid_indices]
ratings_valid=ratings[valid_indices]
dates_valid=dates[valid_indices]

users=users[train_indices]
items=items[train_indices]
ratings=ratings[train_indices]
dates=dates[train_indices]

### Latent Factor Model

In [ ]:
n_users, n_items = len(userIds), len(itemIds)

mean = ratings.mean()
user_bias = torch.zeros(n_users, requires_grad=True, device=device)
item_bias = torch.zeros(n_items, requires_grad=True, device=device)
user_emb = torch.normal(0, 0.1,size=(n_users,10),requires_grad=True, device=device)
item_emb = torch.normal(0, 0.1,size=(n_items,10),requires_grad=True, device=device)

def predict(users, items):
  h = mean + user_bias[users] + item_bias[items]
  h += (user_emb[users]*item_emb[items]).sum(dim=1)
  return h

optim = torch.optim.Adam([user_bias, item_bias, user_emb, item_emb],
                         lr = 0.01, weight_decay=0.00001)

mse = torch.nn.MSELoss()

for epoch in range(1000):
  h = predict(users ,items)
  cost = mse(h, ratings)

  optim.zero_grad()
  cost.backward()
  optim.step()

  if epoch % 100 == 0:
    h_valid = predict(users_valid, items_valid)
    cost_valid = mse(h_valid, ratings_valid)

    print(f"epoch: {epoch}, train mse:{cost.item()}, test mse: {cost_valid.item()}")


##Temporal Item Bias (binning)

In [ ]:
n_bins = 30
tmax = max(dates.max(), dates_valid.max())
date_bins = (dates*n_bins//(tmax+1))
date_bins_valid = (dates_valid*n_bins // (tmax+1))


# Temporal Item Bias
item_bin_bias = torch.zeros(n_items, n_bins, device=device, requires_grad=True)

def predict(users, items, date_bins):
    h = mean + user_bias[users] + item_bias[items]
    h += (user_emb[users] * item_emb[items]).sum(dim=1)
    h += item_bin_bias[items, date_bins]

    return h

##Temporal User Bias
Drifting bias

In [ ]:
tu = users.bincount(dates)/users.bincount()
dev = (dates-tu[users]).sign() * ((dates - tu[users]).abs()**0.4)
dev_valid = (dates_valid - tu[users_valid]).sign()*((dates_valid - tu[users_valid]).abs()**0.4)

# Temporal User Bias
dev_bias = torch.zeros(n_users, device=device, requires_grad=True)

def predict(users, items, date_bins, dev):
    h = mean + user_bias[users] + item_bias[items]
    h += (user_emb[users] * item_emb[items]).sum(dim=1)
    h += item_bin_bias[items, date_bins]
    h += dev_bias[users] * dev[users]

    return h


Per-day user bias

In [ ]:
all_users=torch.cat((users,users_valid))
all_dates=torch.cat((dates,dates_valid))
_,all_userdates=torch.stack((all_users,all_dates)).T.unique(dim=0,return_inverse=True)
userdates=all_userdates[:len(users)]
userdates_valid=all_userdates[len(users):]
n_userdates=all_userdates.max()+1


# Per-day User Bias
perday_bias=torch.zeros(n_userdates,requires_grad=True,device=device)

def predict(users,items,date_bins,dev,userdates):
  h= mean + user_bias[users]+item_bias[items]
  h+=(user_emb[users]*item_emb[items]).sum(dim=1)
  h+=item_bin_bias[items,date_bins]
  h+=dev_bias[users]*dev[users]
  h+=perday_bias[userdates]
  return h